<a href="https://colab.research.google.com/github/Mary-S2/COM-PAPER-REVIEW/blob/main/loadandexplore.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import files
uploaded = files.upload()

Saving BBC News Train.csv.zip to BBC News Train.csv.zip


In [3]:
import pandas as pd

df = pd.read_csv('BBC News Train.csv.zip')
print(df.shape)
print(df.columns.tolist())
print(df['Category'].value_counts())

(1490, 3)
['ArticleId', 'Text', 'Category']
Category
sport            346
business         336
politics         274
entertainment    273
tech             261
Name: count, dtype: int64


In [6]:
print(df['Text'][0][:600])

worldcom ex-boss launches defence lawyers defending former worldcom chief bernie ebbers against a battery of fraud charges have called a company whistleblower as their first witness.  cynthia cooper  worldcom s ex-head of internal accounting  alerted directors to irregular accounting practices at the us telecoms giant in 2002. her warnings led to the collapse of the firm following the discovery of an $11bn (£5.7bn) accounting fraud. mr ebbers has pleaded not guilty to charges of fraud and conspiracy.  prosecution lawyers have argued that mr ebbers orchestrated a series of accounting tricks at 


In [7]:
df['word_count'] = df['Text'].str.split().str.len()
print(df['word_count'].describe())
print("Missing text:", df['Text'].isna().sum())
print("Duplicate texts:", df['Text'].duplicated().sum())

count    1490.000000
mean      385.012752
std       210.898616
min        90.000000
25%       253.000000
50%       337.000000
75%       468.750000
max      3345.000000
Name: word_count, dtype: float64
Missing text: 0
Duplicate texts: 50


In [8]:
import re

# 1. Do any duplicate texts have conflicting labels? (1 = labels agree)
dups = df[df['Text'].duplicated(keep=False)]
print("Max distinct labels per duplicated text:", dups.groupby('Text')['Category'].nunique().max())

# 2. Drop duplicates, keeping the first copy
df = df.drop_duplicates(subset='Text').reset_index(drop=True)
print("Shape after dedupe:", df.shape)

# 3. Collapse repeated spaces/newlines into single spaces
df['clean_text'] = df['Text'].apply(lambda t: re.sub(r'\s+', ' ', t).strip())
print(df['clean_text'][0][:300])

Max distinct labels per duplicated text: 1
Shape after dedupe: (1440, 4)
worldcom ex-boss launches defence lawyers defending former worldcom chief bernie ebbers against a battery of fraud charges have called a company whistleblower as their first witness. cynthia cooper worldcom s ex-head of internal accounting alerted directors to irregular accounting practices at the u


In [9]:
def chunk_text(text, chunk_size=150, overlap=30):
    words = text.split()
    step = chunk_size - overlap
    chunks = []
    for start in range(0, len(words), step):
        piece = words[start:start + chunk_size]
        chunks.append(" ".join(piece))
        if start + chunk_size >= len(words):
            break
    return chunks

rows = []
for _, r in df.iterrows():
    for i, c in enumerate(chunk_text(r['clean_text'])):
        rows.append({
            'chunk_id': f"{r['ArticleId']}_{i}",
            'ArticleId': r['ArticleId'],
            'Category': r['Category'],
            'chunk_text': c,
        })

chunks_df = pd.DataFrame(rows)
print(chunks_df.shape)
print(chunks_df['chunk_text'].str.split().str.len().describe())
print(chunks_df.iloc[0]['chunk_id'], '->', chunks_df.iloc[0]['chunk_text'][:200])

(4981, 4)
count    4981.000000
mean      132.692431
std        33.110327
min        31.000000
25%       136.000000
50%       150.000000
75%       150.000000
max       150.000000
Name: chunk_text, dtype: float64
1833_0 -> worldcom ex-boss launches defence lawyers defending former worldcom chief bernie ebbers against a battery of fraud charges have called a company whistleblower as their first witness. cynthia cooper wo


In [10]:
chunks_df.to_csv('bbc_chunks.csv', index=False)

from google.colab import files
files.download('bbc_chunks.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>